In [0]:
from pyspark.sql import functions as F

def normalize_columns(df):
    """Standardizes column names: lowercase, spaces to underscores."""
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_"))
    return df

In [0]:
df_survey = spark.table("workspace.bronze.customer_survey")
df_survey = normalize_columns(df_survey)

rating_cols = [
    "delivered_on_time_rating", "work_quality_rating",
    "cleanliness_rating", "communication_rating", "overall_satisfaction_rating"
]

df_survey = (
    df_survey
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .dropDuplicates()
)

for c in rating_cols:
    df_survey = df_survey.withColumn(c, F.col(c).cast("int"))


df_survey = df_survey.withColumn(
    "responded_flag",
    F.when(F.col("responded_flag").isNull() & F.col("survey_response_date").isNotNull(), F.lit(True))
    .when(F.col("responded_flag").isNull() & F.col("survey_response_date").isNull(), F.lit(False))
    .otherwise(F.col("responded_flag"))
)

df_order = spark.table("workspace.silver.order").select("order_id", "store_id")
df_survey = df_survey.join(df_order, on="order_id", how="left")


store_avg = (
    df_survey.filter(F.col("responded_flag") == True)
    .groupBy("store_id")
    .agg(*[F.round(F.avg(c)).cast("int").alias(f"{c}_avg") for c in rating_cols])
)

global_avg = (
    df_survey.filter(F.col("responded_flag") == True)
    .agg(*[F.round(F.avg(c)).cast("int").alias(f"{c}_global") for c in rating_cols])
)


df_survey = df_survey.join(store_avg, on="store_id", how="left")
df_survey = df_survey.crossJoin(global_avg)

for c in rating_cols:
    df_survey = df_survey.withColumn(
        c, F.coalesce(F.col(c), F.col(f"{c}_avg"), F.col(f"{c}_global"))
    )

cols_to_drop = [f"{c}_avg" for c in rating_cols] + [f"{c}_global" for c in rating_cols] + ["store_id"]
df_survey = df_survey.drop(*cols_to_drop)

df_survey.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.silver.customer_survey")
print(f"silver.customer_survey: {df_survey.count()} rows written")